# RoboCasa GR1 Tabletop — 下载 + 预览 (Download & Preview)

一键下载并预览 [RoboCasa GR1 Tabletop Tasks](https://github.com/robocasa/robocasa-gr1-tabletop-tasks) 的 **现成场景、官方数据集、官方微调 checkpoint**。

_One-click download & preview of the RoboCasa GR1 Tabletop tasks: ready-made scenes, official datasets, and an official fine-tuned checkpoint._

**这是什么 / What it is**：GR-1 **人形机器人**(Fourier 灵巧手 + 可动腰)做 **24 个桌面拾放任务**。这是 NVIDIA GR00T N1 论文(arXiv 2503.14734)的官方仿真 benchmark。

**和厨房页 `RoboCasa.ipynb` 的区别 / Difference from the kitchen notebook**：
| | 厨房 `RoboCasa.ipynb` | 本页 `RoboCasa-Tabletop.ipynb` |
|---|---|---|
| 机体 | PandaOmron 单臂移动机 | **GR-1 人形 + Fourier 手** |
| 任务 | 厨房原子任务(开柜/水龙头…) | **24 桌面 PnP** |
| repo | `dependencies/robocasa` | `dependencies/robocasa-gr1-tabletop-tasks`(独立 fork) |
| env | `robocasa` / `robocasa_gr00t` | **`robocasa_gr1`**(独立，包名冲突必须隔离) |
| 预览 | 实时 mujoco viewer(需 DISPLAY) | **离屏渲染 mp4 + 数据集 clip 内嵌**(headless 友好) |

**机制 / Mechanism**：逻辑全在 `scripts/install_robocasa_gr1_env.sh` + `scripts/robocasa_gr1_demo.py`，notebook 只 `!bash` / `!python` 调用。

**前置 / Prereqs**：`conda` 可用；§2 数据集预览 **无需任何环境/GPU**(直接下 HF mp4)；§1 装环境 ~20GB；§3 checkpoint 7.6GB；§4/§5 渲染与评测 **需 GPU**。

---
## ⚠️ 当前机器状态 / Current machine state

本机 **4090 正在跑 OpenCabinet(MimicGen 混训)**，GPU 已占满。

- ✅ **可现在做**(纯 CPU/网络，和训练并行不抢卡)：§1 装环境、§2 数据集 clip 预览、§3 下 checkpoint
- ⛔ **暂缓**(需 GPU，等训练结束再跑)：§4 离屏渲染、§5 策略评测

§4/§5 的 cell 已写好但**先不要执行**。

---
## 1. 安装 GR1 Tabletop 环境 (Setup) — 下载 + 安装

创建独立 conda env **`robocasa_gr1`**(python 3.10)，按上游官方配方装 Isaac-GR00T(current main) + robosuite + GR1 repo，并下载 tabletop DigitalCousin 资产。

- 为什么独立 env：GR1 repo 也装一个叫 `robocasa` 的包(v0.2.0，pin `numpy==1.26.4`/`mujoco==3.2.6`)，**和厨房 robocasa 包冲突**，必须隔离。
- `flash-attn` **默认跳过**(只有 §5 评测才需要，它的编译是 ~15min CPU/RAM 高峰，会干扰正在跑的训练)。等 GPU 空了、跑 §5 前再 `BUILD_FLASH_ATTN=1` 补装。
- 幂等：装过/下过会跳过。首次 **15-30 min**(取决于带宽)。

_Dedicated `robocasa_gr1` env (upstream recipe). flash-attn skipped by default — build it before §5 with `BUILD_FLASH_ATTN=1`._

In [ ]:
# 下载 + 安装（纯 CPU/网络，可与训练并行）。首次 15-30 min，幂等。
!bash scripts/install_robocasa_gr1_env.sh

In [ ]:
# 检查环境/repo/资产/checkpoint/clip 状态
!python scripts/robocasa_gr1_demo.py status

In [ ]:
# 24 个桌面任务清单（6 个 core PnP-Close + 18 个 Posttrain novel）
!python scripts/robocasa_gr1_demo.py list

---
## 2. 场景预览 — 数据集 demo clip (Scene Preview, **无需环境/GPU**)

最轻量的预览：官方 [Teleop-Sim 数据集](https://huggingface.co/datasets/nvidia/PhysicalAI-Robotics-GR00T-Teleop-Sim) 的 **LeRobot 子集**里，每个任务每条 demo 都存了一段 **ego_view mp4**(每个仅 ~0.1-0.8MB)。直接从 HF 下**单个文件**就能看真实遥操作演示，**不用装环境、不用渲染、不用 GPU**。

_Lightest preview: download a single ~0.3MB ego-view demo mp4 straight from the HF LeRobot dataset. No env, no render, no GPU._

> ⚠️ 数据集 license = **CC-BY-NC-4.0**(禁商用)。

In [ ]:
# 下载一段 demo clip 并内嵌预览（改 TASK 看别的任务；--episode 0..999 换演示）
import subprocess
from IPython.display import Video, display

TASK = "PnPCupToDrawerClose"   # 用编号也行，如 "1"；或唯一子串 "Cup"
r = subprocess.run(["python", "scripts/robocasa_gr1_demo.py", "clip", TASK, "--episode", "0"],
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr[-500:] if r.returncode else "")
if r.returncode == 0:
    path = r.stdout.strip().splitlines()[-1]
    display(Video(path, embed=True, width=480))

In [ ]:
# 一次性抓 6 个 core 任务的 clip，拼成画廊预览
import subprocess
from IPython.display import Video, display, Markdown

CORE = ["PnPCupToDrawerClose","PnPPotatoToMicrowaveClose","PnPMilkToMicrowaveClose",
        "PnPBottleToCabinetClose","PnPWineToCabinetClose","PnPCanToDrawerClose"]
for t in CORE:
    r = subprocess.run(["python","scripts/robocasa_gr1_demo.py","clip",t,"--episode","0"],
                       capture_output=True, text=True)
    if r.returncode == 0:
        display(Markdown(f"**{t}**"))
        display(Video(r.stdout.strip().splitlines()[-1], embed=True, width=420))
    else:
        print(t, "FAILED:", r.stderr[-200:])

---
## 3. 下载 Checkpoint (Download Model)

GR1 tabletop 有**现成可用的官方微调权重**(不像很多 benchmark 只给基座)：

| key | repo | 体积 | 说明 |
|---|---|---|---|
| ⭐ `youliangtan` | `youliangtan/gr00t-n1.5-robocasa-tabletop-posttrain` | 7.6 GB | **GR00T 论文作者 You Liang Tan** 放的，N1.5 在这 24 任务上 post-train，就是 README 报的 ~47%。结构标准可直接推理(无 model card) |
| `base` | `nvidia/GR00T-N1.5-3B` | ~7 GB | 基座，zero-shot ~42% |
| `n16` | `karthikpythireddi93/gr00t-n16-gr1-tabletop-sft` | ~9.7 GB | N1.6 社区 SFT(跳过 optimizer) |

下到 `~/.cache/robocasa_gr1/checkpoints/`。纯网络，可与训练并行。

_GR1 tabletop has a ready-to-use official fine-tune (`youliangtan`, ~47%), not just a base model._

In [ ]:
# 下主推 checkpoint（GR00T 作者的 N1.5 post-train，7.6GB）。纯网络。
!python scripts/robocasa_gr1_demo.py download-ckpt youliangtan

In [ ]:
# 备选：基座 N1.5（zero-shot 42% 对照）/ N1.6 社区 SFT —— 需要时再下
# !python scripts/robocasa_gr1_demo.py download-ckpt base
# !python scripts/robocasa_gr1_demo.py download-ckpt n16

---
## 4. 场景预览 — 离屏渲染 (Render Preview) ⛔ 需 GPU，训练中先不跑

用 GR1 repo 的 `demo_task.py` 让机器人在场景里发**随机动作**，离屏渲染成 mp4(`MUJOCO_GL=egl` headless)。这验证 **env + 资产装好了**，看到的是你本地新渲染的场景(随机动作，非成功演示——成功演示看 §2)。

_Offscreen-render the scene with random actions (proves the env works). Needs GPU — skip while training runs._

In [ ]:
# ⛔ 需 GPU。本机训练中先别执行。等训练结束、GPU 空了再跑。
import subprocess
from IPython.display import Video, display

TASK = "PnPCupToDrawerClose"
r = subprocess.run(["python","scripts/robocasa_gr1_demo.py","render",TASK],
                   capture_output=True, text=True)
print(r.stdout[-1000:]); print(r.stderr[-500:] if r.returncode else "")
if r.returncode == 0:
    display(Video(r.stdout.strip().splitlines()[-1], embed=True, width=480))

---
## 5. 策略评测 — GR00T 闭环 rollout (Policy Eval) ⛔ 需 GPU，训练中先不跑

> ⚠️ **两套配方，别混用**（我已核对上游两个 README）：
>
> **A. youliangtan N1.5 ckpt（本页 §3 主推）** → 配 **N1.5 时代**配方：server `inference_service.py --data-config fourier_gr1_arms_waist`，embodiment `gr1`。这是 `robocasa-gr1-tabletop-tasks` 原 repo README 的路子。本仓 `scripts/robocasa_gr1_demo.py eval` 走的就是这套。
>
> **B. N1.7 官方权威配方**（Isaac-GR00T `examples/robocasa-gr1-tabletop-tasks/README.md`）→ 用 **uv 独立环境** + `ROBOCASA_GR1_TABLETOP` embodiment：
> ```bash
> # 一次性装 eval sim 环境（uv venv，非本页 conda env）
> bash dependencies/Isaac-GR00T-gr1/gr00t/eval/sim/robocasa-gr1-tabletop-tasks/setup_RoboCasaGR1TabletopTasks.sh
> # Terminal 1 - server
> uv run python gr00t/eval/run_gr00t_server.py --model-path <ckpt> \
>     --embodiment-tag ROBOCASA_GR1_TABLETOP --use-sim-policy-wrapper
> # Terminal 2 - client
> gr00t/eval/sim/robocasa-gr1-tabletop-tasks/robocasa_uv/.venv/bin/python gr00t/eval/rollout_policy.py \
>     --n-episodes 10 --policy-client-host 127.0.0.1 --policy-client-port 5555 \
>     --max-episode-steps 720 --n-action-steps 8 --n-envs 5 \
>     --env-name gr1_unified/PnPBottleToCabinetClose_GR1ArmsAndWaistFourierHands_Env
> ```
> 配方 B 需要一个 **N1.7 `ROBOCASA_GR1_TABLETOP` finetune** 权重（HF 上暂无现成的，官方 benchmark **均值 44.5%**/24 任务 是用它跑的）。youliangtan 是 N1.5，用配方 A。

**前置**：① §3 已下 ckpt ② 装 flash-attn：`BUILD_FLASH_ATTN=1 bash scripts/install_robocasa_gr1_env.sh` ③ GPU 空闲。

_Two eval recipes — A (N1.5, fourier_gr1_arms_waist, for the youliangtan ckpt) vs B (N1.7 authoritative, uv venv + ROBOCASA_GR1_TABLETOP, official avg 44.5%). Skip while training runs._

In [ ]:
# ⛔ 需 GPU + flash-attn。本机训练中先别执行。
# 配方 A（N1.5 + youliangtan ckpt）。先补 flash-attn（GPU 空闲时）：
# !BUILD_FLASH_ATTN=1 bash scripts/install_robocasa_gr1_env.sh
import subprocess
TASK = "PnPCupToDrawerClose"
r = subprocess.run(["python","scripts/robocasa_gr1_demo.py","eval",TASK,
                    "--n-episodes","10","--n-envs","5"],
                   capture_output=True, text=True)
print(r.stdout[-2000:]); print(r.stderr[-800:] if r.returncode else "")

---
## 6. 数据集参考 (Datasets Reference)

| 数据集 | 内容 | 体积 | License |
|---|---|---|---|
| [`nvidia/PhysicalAI-Robotics-GR00T-Teleop-Sim`](https://huggingface.co/datasets/nvidia/PhysicalAI-Robotics-GR00T-Teleop-Sim) | 24 任务遥操作 demo(本页 §2 预览源) | **55.4 GB**(HDF5 14GB + LeRobot 41.5GB) | ⚠️ CC-BY-NC-4.0 |
| [`nvidia/PhysicalAI-Robotics-GR00T-X-Embodiment-Sim`](https://huggingface.co/datasets/nvidia/PhysicalAI-Robotics-GR00T-X-Embodiment-Sim) | 跨机体 post-training 池 | 240k 轨迹 | CC-BY-4.0 |

§2 只下单个 mp4 文件做预览。若要**完整数据集**(训练用)：

```bash
# 单任务 HDF5（~440-816MB/任务，回放/检查用）
conda run -n robocasa_gr1 python - <<'PY'
from huggingface_hub import hf_hub_download
hf_hub_download("nvidia/PhysicalAI-Robotics-GR00T-Teleop-Sim", repo_type="dataset",
               filename="HDF5/PnPCupToDrawerClose.hdf5",
               local_dir="~/.cache/robocasa_gr1/datasets")
PY

# 完整 LeRobot 子集（41.5GB，GR00T 训练格式）
# huggingface-cli download nvidia/PhysicalAI-Robotics-GR00T-Teleop-Sim --repo-type dataset \
#   --include "LeRobot/*" --local-dir ~/.cache/robocasa_gr1/datasets
```

---
## 7. 说明 (Notes)

- 本页是**独立预览线**，和 OpenCabinet 主线(GR1 人形 vs 单臂移动机)互不相干，不碰主线训练。
- §2 是**最稳的预览**(纯 HF 文件，无环境依赖)；§4 渲染依赖 env+资产+GPU；§5 评测再加 ckpt+flash-attn。
- `youliangtan` checkpoint 无 model card，但作者权威(GR00T 论文作者)、文件结构标准，可直接推理。
- 数据集 Teleop-Sim 是 **CC-BY-NC**(禁商用)，注意合规。